Import needed libraries and confirm our env vars

In [ ]:
from utils import setup_notebook

if not setup_notebook(
    required_keys=[
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
        "AWS_DEFAULT_REGION",
    ],
):
    raise ValueError("Failed to setup notebook, please check your .env file")

In [ ]:
import os
import pprint

import boto3
from types_boto3_bedrock_runtime.type_defs import ToolConfigurationTypeDef

Setup the Bedrock client

In [ ]:
bedrock_runtime = boto3.client(
    "bedrock-runtime",
    region_name="us-east-1",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),
)

Create a basic tool for Anthropic to call

In [ ]:
def get_calculator_tool() -> ToolConfigurationTypeDef:
    """
    Returns a tool configuration for a basic calculator that can be used with Bedrock.
    """
    return {
        "tools": [
            {
                "toolSpec": {
                    "name": "calculator",
                    "description": "Perform basic arithmetic calculations",
                    "inputSchema": {
                        "json": {
                            "type": "object",
                            "properties": {
                                "operation": {
                                    "type": "string",
                                    "enum": ["add", "subtract", "multiply", "divide"],
                                },
                                "numbers": {
                                    "type": "array",
                                    "items": {"type": "number"},
                                    "minItems": 2,
                                },
                            },
                            "required": ["operation", "numbers"],
                        },
                    },
                },
            },
        ],
    }

Define functions to stream and parse Bedrock responses

In [ ]:
SEPARATOR = "-" * 50


def stream_response(bedrock_runtime, model_id, input_text, max_tokens=100):
    """
    Streams responses from Bedrock using the converse_stream API.
    Displays raw JSON events as they arrive.
    """

    try:
        response = bedrock_runtime.converse_stream(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": input_text}]}],
            inferenceConfig={
                "maxTokens": max_tokens,
            },
            toolConfig=get_calculator_tool(),
        )

        print("=== Starting Stream ===\n")

        print("RESPONSE OBJECT")
        pprint.pprint(response)
        print(SEPARATOR)

        # Process the streaming response
        for event in response["stream"]:
            # Decode bytes to string and parse JSON
            print("NEW CHUNK")
            pprint.pprint(event)
            print(SEPARATOR)

    except Exception as e:
        print(f"\nError during streaming: {e!s}")
        return

    print("\n=== Stream Complete ===")

Use

In [ ]:
model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
stream_response(bedrock_runtime, model_id, "Will you please add 2 and 2 together?")